# Hate Crime Bias Classification  
### Before & After Data Cleaning — Model Comparison Notebook

This notebook demonstrates how data cleaning, enrichment, and restructuring
significantly impact machine learning performance.

We compare two models:

1. **BEFORE MODEL** — trained on *raw, uncleaned* incident-level data  
2. **AFTER MODEL** — trained on *cleaned, enriched, offense-level* dataset

The target variable is:

**Race/Ethnicity Bias (1) vs Non-Race Bias (0)**

---


## 1. Datasets Used

### 🔴 Before Cleaning (Dirty Data)
- File: `data/interim/ir_decoded.parquet`
- One row = one incident
- Contains 10 offense slots, most empty
- Missing values not handled
- No unpivoting  
- No population enrichment

### 🟢 After Cleaning (Processed Data)
- File: `data/processed/hatecrimes_enriched.parquet`
- One row = one **offense**
- Cleaned categorical / numeric fields
- Missing values handled
- Enriched with:
  - city population
  - county population
- Offense slots collapsed using unpivoting
- Additional engineered features included

---


In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score, average_precision_score

import warnings
warnings.filterwarnings('ignore')


# 2. BEFORE MODEL — Training on Raw Data

This model uses the **raw incident-level dataset**, without any cleaning,
feature engineering, or offense unpivoting.

This is intentionally done to illustrate how poor data quality can ruin model performance.


In [18]:
# Load raw data
df_raw = pd.read_parquet("../data/interim/ir_decoded.parquet")

bias_cols = [
    "bias_1a_category",
    "bias_1b_category",
    "bias_1c_category",
    "bias_1d_category",
    "bias_1e_category"
]
# Combine into a single column
df_raw["bias_category"] = (
    df_raw[bias_cols]
    .bfill(axis=1)     # take the first non-null value
    .iloc[:, 0]
)
# Target: Race/Ethnicity vs Non-Race bias
df_raw["target_bias"] = df_raw["bias_category"].eq("Race/Ethnicity").astype(int)

# Select simple features
features = ["total_victims", "total_offenders", "incident_date"]
X = df_raw[features]
y = df_raw["target_bias"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Fit model
model_dirty = LogisticRegression(max_iter=200)
model_dirty.fit(X_train, y_train)

# Predict
pred_dirty = model_dirty.predict(X_test)
y_prob = model_dirty.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

print("=== DIRTY MODEL RESULTS ===")
print(classification_report(y_test, pred_dirty))
print(f"\nROC AUC: {auc:.2f}")
print(f"PR AUC: {pr_auc:.2f}")


=== DIRTY MODEL RESULTS ===
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      5124
           1       0.56      1.00      0.72      6545

    accuracy                           0.56     11669
   macro avg       0.28      0.50      0.36     11669
weighted avg       0.31      0.56      0.40     11669


ROC AUC: 0.45
PR AUC: 0.53


# 3. AFTER MODEL — Training on Cleaned & Enriched Offense-Level Data

This model uses the fully processed dataset created by:

- Removing invalid or empty offense slots  
- Unpivoting into one-row-per-offense  
- Cleaning categorical inconsistencies  
- Converting population fields  
- Adding engineered features  
- Standardizing numeric features

This dataset provides much more signal for classification.


In [19]:
# ======================================================
# 1. Load Data & Define Columns
# ======================================================
df = pd.read_parquet("../data/processed/hatecrimes_enriched.parquet")

bias_cols = [
    "bias_a_category", "bias_b_category", "bias_c_category",
    "bias_d_category", "bias_e_category"
]

# Create target
df["bias_category"] = df[bias_cols].bfill(axis=1).iloc[:, 0]
df = df.dropna(subset=["bias_category"])
df["target_race_bias"] = df["bias_category"].eq("Race/Ethnicity").astype(int)

categorical_cols = [
    "offense", "location", "day_of_week", "state_name", "offender_race"
]

numeric_cols = [
    "total_victims", "total_offenders",
    "year", "month", "is_weekend", "state_population"
]

# Explicit copy to avoid SettingWithCopyWarning
X = df[categorical_cols + numeric_cols].copy()
y = df["target_race_bias"]

# Ensure numeric columns are float
X[numeric_cols] = X[numeric_cols].astype("float64")

# ======================================================
# 2. Split Data (Before any processing to prevent leakage)
# ======================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# ======================================================
# 3. Define Pipelines
# ======================================================

# Numeric Pipeline:
# 1. Impute missing values (and add binary indicator column for missingness)
# 2. Scale the data
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean', add_indicator=True)), 
    ('scaler', StandardScaler())
])

# Categorical Pipeline:
# 1. Handle missing categoricals (optional, but good practice)
# 2. One Hot Encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine them
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

# ======================================================
# 4. Full Model Pipeline
# ======================================================
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=300,
        max_depth=20,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1 
    ))
])

# ======================================================
# 5. Train and Evaluate
# ======================================================
# Note: No manual imputation here. The pipeline handles it safely.
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

print("=== CLEAN MODEL RESULTS ===")
print(classification_report(y_test, y_pred))
print(f"\nROC AUC: {auc:.2f}")
print(f"PR AUC: {pr_auc:.2f}")

=== CLEAN MODEL RESULTS ===
              precision    recall  f1-score   support

           0       0.67      0.40      0.50      5297
           1       0.65      0.84      0.73      6872

    accuracy                           0.65     12169
   macro avg       0.66      0.62      0.62     12169
weighted avg       0.66      0.65      0.63     12169


ROC AUC: 0.69
PR AUC: 0.72


# 🔍 4. Model Performance Comparison: Dirty vs Clean Dataset

### **Side-by-Side Metrics**

| **Metric**                  | **Dirty Model** | **Clean Model** | **Interpretation**                                                                 |
| ----------------------- | --------------- | --------------- | ---------------------------------------------------------------------------------- |
| **Accuracy**            | 0.56            | **0.65**        | Clean data improves overall correctness by **+9 percentage points**                |
| **Precision (Class 0)** | 0.00            | **0.67**        | Dirty model fails to identify Class 0 entirely. Clean model fixes this.            |
| **Recall (Class 0)**    | 0.00            | **0.40**        | Clean data allows the model to recover missing Class 0 patterns.                   |
| **F1 (Class 0)**        | 0.00            | **0.50**        | Major improvement in minority class performance.                                   |
| **Precision (Class 1)** | 0.56            | **0.65**        | Slight improvement; cleaner features reduce noise.                                 |
| **Recall (Class 1)**    | 1.00            | **0.84**        | Dirty model predicts almost everything as Class 1; clean model is more balanced.   |
| **F1 (Class 1)**        | 0.72            | **0.73**        | Similar performance, but cleaner model does so *without collapsing other classes*. |
| **Macro F1**            | 0.36            | **0.62**        | Balanced performance improves dramatically (+25 points).                           |
| **Weighted F1**         | 0.40            | **0.63**        | Clean model handles class imbalance much better.                                   |
| **ROC AU**    | 0.45  | **0.69**  | Clean model can actually rank classes correctly |
| **PR AU**    | 0.53  | **0.72**  | Big improvement in precision-recall tradeoff |

---
### 🔍 Key Observations

**1. Dirty Model Completely Failed on Class 0**

- Precision, recall, F1 for **class 0 all equal 0.00**.
- The model predicts **every sample as class 1**.
- High recall (1.00) for class 1 is misleading because it reflects **model collapse due to imbalanced or noisy features**.

This shows that the dirty dataset did NOT contain usable signal for the model.

**2. Clean Model Learns Both Classes**

After data cleaning:
- **Class 0 precision improves from 0.00 → 0.67**.
- **Class 0 F1 improves from 0.00 → 0.50**.
- The model no longer collapses into predicting a single class.
- Class 1 performance remains strong.

This indicates that proper preprocessing enabled the model to detect patterns in both classes.

**3. Macro F1-Score Improves Significantly**

| Metric       | Dirty | Clean    | Change    |
| ------------ | ----- | -------- | --------- |
| **Macro F1** | 0.36  | **0.62** | **+0.26** |

Macro F1 treats both classes equally → this improvement shows substantial gains in balanced performance.

**4. Accuracy Slightly Improves, But It’s Not the Main Story**

Accuracy: **0.56 → 0.65**

The improvement reflects better predictions on both classes. The small improvement is expected, because the dirty model’s accuracy was artificially inflated by always predicting the majority class.

The real improvement is in learning the minority class (class 0).

**5. Macro performance improves significantly**

* Macro Precision: **0.28 → 0.66**
* Macro Recall: **0.50 → 0.62**
* Macro F1: **0.36 → 0.62**

This confirms **fairness, stability, and balanced learning**, which is essential in hate-crime classification.

**6. AUC improves greatly**

| Metric       | Dirty | Clean    | Change    |
| ------------ | ----- | -------- | --------- |
| **ROC AUC**  | 0.45  | **0.69** | **+0.24** |
| **PR AUC**   | 0.53  | **0.72** | **+0.19** |

The improvement in ROC AUC suggests that clean model can actually rank classes correctly, and a jump of +19 percentage point in PR AUC indicates a big improvement in precision-recall tradeoff .

---

## ✅ **Final Conclusion**

Cleaning the dataset led to a **substantial improvement** in model quality:

* Eliminated the collapse into predicting only one class
* Recovered meaningful patterns for Class 0
* Increased accuracy by **+9%**
* Improved macro F1 by **+25 points**, demonstrating a **far more balanced** and **reliable** model
* Improved ROC AUC and PR AUC by **+24 points** and **+19 points** means the model finally learned real differences between Race-bias vs Non-race-bias incidents.


**The Clean Model is significantly better and is the correct version to use for the project.**
